# 03 Annual Trend Analysis

This notebook reads `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv`, uses `award_year` to calculate the complete 2010-2023 annual distribution and shares, and exports the Fig. 1 annual-growth chart, annual distribution table, and draft Results paragraph.

Running the full notebook writes `output/figures/Fig1_annual_growth.*`, `output/tables/03_year_distribution.csv`, and `output/logs/03_year_trend_text.md`; this update only modifies the notebook file.


## Chart Design Notes

- Figure contract: Fig. 1 should summarize the temporal concentration, funding-program composition, and approved funding-scale structure of the included NSFC-funded BAE corpus.
- Evidence chain: Panel a uses the `agency_program` project type before the semicolon to show funding-type composition by record count; panel b uses `amount_original` to compare approved funding totals by type; panel c uses the full `award_year` axis from 2010 to 2023 to show temporal concentration.
- Archetype: Quantitative grid with enlarged composition and compressed funding-scale support panels above the temporal bar chart.
- Chart family: Annual bar chart plus funding-type donut and approved-amount horizontal bars. The cumulative line is intentionally removed.
- Data sufficiency: The new master table contains valid `award_year` values for every year from 2010 through 2023. Panel c and the full year-distribution table use the same complete axis. Funding type and approved amount are available for all records.
- Palette policy: Panel c annual bars use a neutral grey, while panel a uses a restrained multi-hue pastel family for funding types and panel b matches those funding-type colors.
- Output footprint: Static Python chart exported to SVG, PDF, TIFF, and PNG. All visible figure text is English. The figure intentionally has no global top title and no bottom footnote.


In [ ]:
# Import the basic libraries required by this notebook.
# pandas reads and summarizes CSV files; numpy supports numeric checks; matplotlib/seaborn export static charts.
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms

# Register and require Times New Roman; stop immediately instead of using a fallback font if it is unavailable.
TIMES_NEW_ROMAN_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
]
for font_path in TIMES_NEW_ROMAN_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont("Times New Roman", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("Times New Roman is required for figure export but was not found by matplotlib.") from exc

import matplotlib.ticker as mticker
import seaborn as sns

# Set vector-figure font policy so SVG/PDF preserve editable text where possible instead of converting text to paths.
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

DATA_FILENAME = "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
YEAR_AXIS = list(range(2010, 2024))

# Search upward from the current directory for the project root so the notebook works from either the project root or code/.
def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "data" / DATA_FILENAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Cannot locate data/{DATA_FILENAME} from the current working directory.")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / DATA_FILENAME
FIGURE_DIR = PROJECT_ROOT / "output" / "figures"
TABLE_DIR = PROJECT_ROOT / "output" / "tables"
LOG_DIR = PROJECT_ROOT / "output" / "logs"

# Create only the output directories authorized for this task; do not modify data/ or files owned by other subagents.
for directory in [FIGURE_DIR, TABLE_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")

In [ ]:
# Read the new master dataset.
# dtype is not forced so original source fields are preserved; later steps safely process only fields needed for plotting.
df = pd.read_csv(DATA_PATH)

# Basic field validation: this analysis depends on year, funding type, and approved amount.
required_columns = {"award_year", "agency_program", "amount_original", "currency", "amount_type"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

# Convert award_year to numeric years; fail immediately on unparseable years to avoid silently dropping records.
award_year = pd.to_numeric(df["award_year"], errors="coerce")
if award_year.isna().any():
    bad_rows = df.loc[award_year.isna(), ["award_year"]].head(10)
    raise ValueError(f"award_year contains missing or non-numeric values. Examples:\n{bad_rows}")
award_year = award_year.astype(int)

# The current data stores all amounts as approved_amount_10k_cny; convert to numeric values for the lower-right amount panel.
approved_amount_10k_cny = pd.to_numeric(df["amount_original"], errors="coerce")
if approved_amount_10k_cny.isna().any():
    bad_rows = df.loc[approved_amount_10k_cny.isna(), ["amount_original", "currency", "amount_type"]].head(10)
    raise ValueError(f"amount_original contains missing or non-numeric values. Examples:\n{bad_rows}")

amount_type_values = set(df["amount_type"].dropna().astype(str).str.strip())
currency_values = set(df["currency"].dropna().astype(str).str.strip())
assert amount_type_values == {"approved_amount_10k_cny"}, f"Unexpected amount_type values: {sorted(amount_type_values)}"
assert currency_values == {"CNY"}, f"Unexpected currency values: {sorted(currency_values)}"

total_n = int(len(df))
total_approved_amount_million_cny = float(approved_amount_10k_cny.sum() / 100)

# The new master table must fully cover 2010-2023 and must not contain records outside that year axis.
raw_years = sorted(award_year.unique().tolist())
missing_axis_years = sorted(set(YEAR_AXIS).difference(raw_years))
outside_axis_years = sorted(set(raw_years).difference(YEAR_AXIS))
assert not missing_axis_years, f"Missing award_year values on the 2010-2023 axis: {missing_axis_years}"
assert not outside_axis_years, f"award_year values outside 2010-2023: {outside_axis_years}"

print(f"Total records: {total_n}")
print(f"Raw award years: {raw_years}")
print(f"Award-year axis: {YEAR_AXIS[0]}-{YEAR_AXIS[-1]} ({len(YEAR_AXIS)} years)")
print(f"Approved amount total: {total_approved_amount_million_cny:.2f} million CNY")


In [ ]:
# Build the complete year axis; both charts and tables use fixed 2010-2023 coverage rather than shortening to data peaks or plot windows.
year_axis = YEAR_AXIS.copy()
raw_counts = award_year.value_counts().sort_index()

year_distribution = (
    raw_counts
    .reindex(year_axis, fill_value=0)
    .rename_axis("award_year")
    .reset_index(name="n")
)

# Shares use the total new master-table sample as the denominator; the CSV keeps decimals and readable percentage labels for manuscript tables or manual checks.
year_distribution["share"] = year_distribution["n"] / total_n
year_distribution["share_percent"] = (year_distribution["share"] * 100).round(4)
year_distribution["share_label"] = year_distribution["share"].map(lambda value: f"{value * 100:.2f}%")

# Label each year with an analysis period; note is populated only if an empty year appears in the future.
year_distribution["period"] = np.select(
    [
        year_distribution["award_year"].between(2010, 2014),
        year_distribution["award_year"].between(2015, 2019),
    ],
    ["Early period (2010-2014)", "Middle period (2015-2019)"],
    default="Recent period (2020-2023)",
)
year_distribution["raw_year_present"] = year_distribution["award_year"].isin(raw_counts.index)
year_distribution["note"] = np.where(
    year_distribution["raw_year_present"],
    "",
    "No records in source dataset",
)

# Funding-type definition: reuse the overview dashboard rule, i.e. the project type before the semicolon in agency_program.
def classify_agency_program_value(value):
    value = "" if pd.isna(value) else str(value).split(";")[0].strip()
    if "青年" in value:
        return "Young Scientists Fund"
    if "面上" in value:
        return "General Program"
    if "地区" in value:
        return "Regional Fund"
    if "重点" in value:
        return "Key Program"
    if "重大" in value:
        return "Major Research Plan"
    if "联合" in value:
        return "Joint Fund"
    return "Other / not classified"

df = df.copy()
df["funding_type"] = df["agency_program"].map(classify_agency_program_value)
df["approved_amount_10k_cny"] = approved_amount_10k_cny

funding_counts = df["funding_type"].value_counts()
funding_order = [
    "General Program",
    "Young Scientists Fund",
    "Regional Fund",
    "Key Program",
    "Major Research Plan",
    "Joint Fund",
    "Other / not classified",
]
funding_categories = [category for category in funding_order if category in funding_counts.index]
funding_categories.extend([category for category in funding_counts.index if category not in funding_categories])

funding_distribution = (
    funding_counts
    .reindex(funding_categories, fill_value=0)
    .rename_axis("funding_type")
    .reset_index(name="n")
)
funding_distribution = funding_distribution.loc[funding_distribution["n"].gt(0)].reset_index(drop=True)
funding_distribution["share"] = funding_distribution["n"] / total_n
funding_distribution["share_percent"] = (funding_distribution["share"] * 100).round(4)
funding_distribution["share_label"] = funding_distribution["share"].map(lambda value: f"{value * 100:.2f}%")

amount_distribution = (
    df.groupby("funding_type", as_index=False)
    .agg(
        n=("funding_type", "size"),
        approved_amount_10k_cny=("approved_amount_10k_cny", "sum"),
        median_amount_10k_cny=("approved_amount_10k_cny", "median"),
    )
)
amount_distribution["approved_amount_million_cny"] = amount_distribution["approved_amount_10k_cny"] / 100
amount_distribution["amount_share"] = amount_distribution["approved_amount_10k_cny"] / amount_distribution["approved_amount_10k_cny"].sum()
amount_distribution["amount_share_percent"] = (amount_distribution["amount_share"] * 100).round(4)
amount_distribution["amount_share_label"] = amount_distribution["amount_share"].map(lambda value: f"{value * 100:.2f}%")
amount_distribution["funding_type"] = pd.Categorical(amount_distribution["funding_type"], categories=funding_categories, ordered=True)
amount_distribution = amount_distribution.sort_values("funding_type").reset_index(drop=True)

# Summary validation: the year axis must exactly equal 2010-2023, and funding type and amount must cover the full sample.
assert year_distribution["award_year"].tolist() == YEAR_AXIS
assert year_distribution["raw_year_present"].all()
assert int(year_distribution["n"].sum()) == total_n
assert int(funding_distribution["n"].sum()) == total_n
assert np.isclose(amount_distribution["approved_amount_10k_cny"].sum(), approved_amount_10k_cny.sum())

# Export the annual distribution table owned by this subagent. Funding-type and amount distributions are written to the log, and figures are generated directly from this table.
table_path = TABLE_DIR / "03_year_distribution.csv"
year_distribution.to_csv(table_path, index=False, encoding="utf-8-sig")

display(year_distribution)
display(funding_distribution)
amount_distribution


In [ ]:
# Define visual tokens for the static chart.
# Note: variable names and comments may be local, but all title, axis, note, and other visible figure text remains English.
FONT_FAMILY = ["Times New Roman"]
MONO_FONT_FAMILY = ["Times New Roman"]

TOKENS = {
    "surface": "#FFFFFF",
    "panel": "#FFFFFF",
    "ink": "#1F2430",
    "muted": "#6F768A",
    "grid": "#E6E8F0",
    "axis": "#D7DBE7",
}
BLACK = "#000000"

COLOR_FAMILIES = {
    "blue": {"xlight": "#EAF1FE", "light": "#CEDFFE", "base": "#A3BEFA", "mid": "#5477C4", "dark": "#2E4780"},
    "teal": {"xlight": "#EAF7F3", "light": "#BFDCD4", "base": "#8FB9A8", "mid": "#4E8E7D", "dark": "#28675B"},
    "amber": {"xlight": "#FFF6E8", "light": "#ECD6A7", "base": "#D4B483", "mid": "#A57B38", "dark": "#755420"},
    "rose": {"xlight": "#FBEFEF", "light": "#E6C6C3", "base": "#C99993", "mid": "#A8645F", "dark": "#743B37"},
    "violet": {"xlight": "#F3F0FA", "light": "#D8CEE8", "base": "#B5A2CA", "mid": "#8066A8", "dark": "#554174"},
    "neutral": {"xlight": "#F4F5F7", "light": "#E2E5EA", "base": "#C5CAD3", "mid": "#7A828F", "dark": "#464C55"},
}

FUNDING_COLORS = {
    "General Program": COLOR_FAMILIES["blue"]["mid"],
    "Young Scientists Fund": COLOR_FAMILIES["teal"]["base"],
    "Regional Fund": COLOR_FAMILIES["amber"]["base"],
    "Key Program": COLOR_FAMILIES["rose"]["base"],
    "Major Research Plan": COLOR_FAMILIES["violet"]["base"],
    "Joint Fund": COLOR_FAMILIES["neutral"]["mid"],
    "Other / not classified": COLOR_FAMILIES["neutral"]["light"],
}

def use_chart_theme() -> None:
    """Apply the shared chart theme; plot text is still controlled by later English strings."""
    sns.set_theme(
        style="whitegrid",
        rc={
            "figure.facecolor": TOKENS["surface"],
            "figure.edgecolor": "none",
            "savefig.facecolor": TOKENS["surface"],
            "savefig.edgecolor": "none",
            "axes.facecolor": TOKENS["panel"],
            "axes.edgecolor": TOKENS["axis"],
            "axes.labelcolor": TOKENS["ink"],
            "axes.grid": True,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": TOKENS["grid"],
            "grid.linewidth": 0.8,
            "font.family": "serif",
            "font.serif": FONT_FAMILY,
            "font.sans-serif": FONT_FAMILY,
            "font.monospace": MONO_FONT_FAMILY,
            "patch.linewidth": 1.0,
        },
    )

use_chart_theme()


In [ ]:
# Draw Fig. 1: annual bar chart + funding-type composition + approved-amount structure.
# All visible text in the figure is English; retained Chinese appears only in matching rules or source data values.
FIG1C_YEAR_AXIS = year_axis.copy()
plot_df = year_distribution.loc[year_distribution["award_year"].isin(FIG1C_YEAR_AXIS)].copy()
fig1c_n = int(plot_df["n"].sum())
assert fig1c_n == total_n
funding_plot_df = funding_distribution.copy()
amount_plot_df = amount_distribution.sort_values("approved_amount_million_cny", ascending=True).copy()

# Color strategy: panel c annual bars use neutral grey; panels a/b share funding-type colors.
ANNUAL_BAR_COLOR = "#8A8F98"
bar_color = ANNUAL_BAR_COLOR
bar_edge = ANNUAL_BAR_COLOR
REFERENCE_LINE_COLOR = "#B8BEC8"
AXIS_SPINE_LINEWIDTH = 0.65
funding_colors = [FUNDING_COLORS.get(label, COLOR_FAMILIES["neutral"]["light"]) for label in funding_plot_df["funding_type"]]
amount_colors = [FUNDING_COLORS.get(label, COLOR_FAMILIES["neutral"]["light"]) for label in amount_plot_df["funding_type"].astype(str)]

PANEL_TITLE_SIZE = 11.2
AXIS_LABEL_SIZE = 8.8
TICK_LABEL_SIZE = 7.4
VALUE_LABEL_SIZE = 7.1
CALLOUT_LABEL_SIZE = 7.0
CENTER_LABEL_SIZE = 10.4

fig = plt.figure(figsize=(9.35, 6.05), dpi=160)
PANEL_C_WIDTH_FRACTION = 0.78
PANEL_C_YLABEL_PAD = 11.0
gs = fig.add_gridspec(2, 2, width_ratios=[1.28, 0.82], height_ratios=[1.06, 1.00], wspace=0.34, hspace=0.34)
ax_donut = fig.add_subplot(gs[0, 0])
ax_amount = fig.add_subplot(gs[0, 1])
ax = fig.add_subplot(gs[1, :])
fig.subplots_adjust(left=0.070, right=0.975, top=0.935, bottom=0.108, wspace=0.34, hspace=0.34)
annual_axis_position = ax.get_position()
panel_left_x = annual_axis_position.x0
panel_c_axis_width = annual_axis_position.width * PANEL_C_WIDTH_FRACTION
ax.set_position([
    panel_left_x,
    annual_axis_position.y0,
    panel_c_axis_width,
    annual_axis_position.height,
])

panel_title_style = {
    "ha": "left",
    "va": "bottom",
    "fontsize": PANEL_TITLE_SIZE,
    "fontweight": "bold",
    "color": BLACK,
}

def add_panel_title(axis, label, title, y=1.035, x=0.0, transform=None):
    axis.text(x, y, f"{label} {title}", transform=transform or axis.transAxes, **panel_title_style)

# Panel c: annual counts across the full 2010-2023 award-year axis.
bars = ax.bar(
    plot_df["award_year"],
    plot_df["n"],
    width=0.62,
    color=bar_color,
    edgecolor=bar_edge,
    linewidth=0.8,
    zorder=2,
)

# Keep numeric labels above bars so readers can directly check each annual N.
y_max = max(100, int(np.ceil(plot_df["n"].max() / 100) * 100 + 100))
ax.set_ylim(0, y_max)
for bar, count in zip(bars, plot_df["n"].tolist()):
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    ax.text(
        x,
        y + y_max * 0.025 if count > 0 else y_max * 0.012,
        f"{int(count)}",
        ha="center",
        va="bottom",
        fontsize=VALUE_LABEL_SIZE,
        color=BLACK,
        fontfamily="Times New Roman",
        zorder=4,
    )

ax.set_xlabel("Award year", fontsize=AXIS_LABEL_SIZE, labelpad=5, color=BLACK)
ax.set_ylabel("Annual awards", fontsize=AXIS_LABEL_SIZE, labelpad=PANEL_C_YLABEL_PAD, color=BLACK)
ax.set_xlim(YEAR_AXIS[0] - 0.6, YEAR_AXIS[-1] + 0.6)
ax.set_xticks(plot_df["award_year"])
ax.set_xticklabels(plot_df["award_year"].astype(str), fontsize=TICK_LABEL_SIZE, rotation=0, ha="center", color=BLACK)
ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5, integer=True))
ax.grid(axis="y", color=REFERENCE_LINE_COLOR, linewidth=0.45)
ax.grid(axis="x", visible=False)
ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE, colors=BLACK, length=3, width=0.7)

for spine in ["left", "bottom"]:
    ax.spines[spine].set_color(BLACK)
    ax.spines[spine].set_linewidth(AXIS_SPINE_LINEWIDTH)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
ylabel_bbox_fig = ax.yaxis.label.get_window_extent(renderer=renderer).transformed(fig.transFigure.inverted())
panel_c_content_shift = max(0, panel_left_x - ylabel_bbox_fig.x0)
ax.set_position([
    panel_left_x + panel_c_content_shift,
    annual_axis_position.y0,
    panel_c_axis_width,
    annual_axis_position.height,
])

panel_c_title_transform = mtransforms.blended_transform_factory(fig.transFigure, ax.transAxes)
add_panel_title(ax, "c", "Annual growth by award year", x=panel_left_x, transform=panel_c_title_transform)

# Panel a: funding-type composition. Place the two dominant types on the right and stack smaller-share types vertically on the left.
donut_startangle = 150
large_right_labels = {"General Program", "Young Scientists Fund"}
left_label_y = {
    "Regional Fund": -0.72,
    "Key Program": -0.36,
    "Major Research Plan": 0.00,
    "Joint Fund": 0.36,
    "Other / not classified": 0.72,
}
right_label_y = {
    "General Program": 0.56,
    "Young Scientists Fund": -0.56,
}
wedges, _ = ax_donut.pie(
    funding_plot_df["n"],
    colors=funding_colors,
    startangle=donut_startangle,
    counterclock=False,
    radius=0.99,
    wedgeprops={"width": 0.39, "edgecolor": TOKENS["surface"], "linewidth": 1.0},
)
ax_donut.text(0, 0, f"N={total_n}", ha="center", va="center", fontsize=CENTER_LABEL_SIZE, fontweight="semibold", color=TOKENS["ink"])
ax_donut.set_aspect("equal")
ax_donut.set_axis_off()
donut_title_transform = mtransforms.blended_transform_factory(fig.transFigure, ax_donut.transAxes)
add_panel_title(ax_donut, "a", "Funding type composition", y=1.025, x=panel_left_x, transform=donut_title_transform)

label_specs = []
for wedge, row in zip(wedges, funding_plot_df.itertuples(index=False)):
    theta = np.deg2rad((wedge.theta1 + wedge.theta2) / 2)
    x = float(np.cos(theta))
    y = float(np.sin(theta))
    funding_type = row.funding_type
    side = 1 if funding_type in large_right_labels else -1
    target_y = right_label_y.get(funding_type, left_label_y.get(funding_type, y))
    label_specs.append(
        {
            "funding_type": funding_type,
            "n": int(row.n),
            "share": float(row.share),
            "x": x,
            "y": y,
            "side": side,
            "target_y": target_y,
        }
    )

def distribute_label_positions(specs, min_gap=0.26, lower=-0.82, upper=0.82):
    """Small deterministic collision-avoidance helper for donut labels."""
    if not specs:
        return []
    specs = sorted(specs, key=lambda item: item["target_y"])
    for idx in range(1, len(specs)):
        gap = specs[idx]["target_y"] - specs[idx - 1]["target_y"]
        if gap < min_gap:
            specs[idx]["target_y"] = specs[idx - 1]["target_y"] + min_gap
    overflow = specs[-1]["target_y"] - upper
    if overflow > 0:
        for item in specs:
            item["target_y"] -= overflow
    underflow = lower - specs[0]["target_y"]
    if underflow > 0:
        for item in specs:
            item["target_y"] += underflow
    return specs

for side in [-1, 1]:
    if side < 0:
        side_specs = distribute_label_positions(
            [item.copy() for item in label_specs if item["side"] == side],
            min_gap=0.34,
            lower=-0.78,
            upper=0.78,
        )
        label_x = -1.34
    else:
        side_specs = distribute_label_positions(
            [item.copy() for item in label_specs if item["side"] == side],
            min_gap=0.50,
            lower=-0.62,
            upper=0.62,
        )
        label_x = 1.24
    for item in side_specs:
        xy = (item["x"] * 0.84, item["y"] * 0.84)
        xytext = (label_x, item["target_y"])
        if item["n"] < 20:
            label = f"{item['funding_type']}  {item['n']} ({item['share'] * 100:.1f}%)"
        else:
            label = f"{item['funding_type']}\n{item['n']} ({item['share'] * 100:.1f}%)"
        ax_donut.annotate(
            label,
            xy=xy,
            xytext=xytext,
            ha="left" if side > 0 else "right",
            va="center",
            fontsize=CALLOUT_LABEL_SIZE,
            color=BLACK,
            linespacing=1.04,
            arrowprops={
                "arrowstyle": "-",
                "color": REFERENCE_LINE_COLOR,
                "lw": 0.65,
                "shrinkA": 0,
                "shrinkB": 0,
                "connectionstyle": "arc3,rad=0.12",
            },
            annotation_clip=False,
        )

ax_donut.set_xlim(-1.82, 1.82)
ax_donut.set_ylim(-1.15, 1.15)

# Panel b: approved funding scale by type. Keep total-amount colors aligned with the funding-type colors in panel a.
y_pos = np.arange(len(amount_plot_df))
amount_bars = ax_amount.barh(
    y_pos,
    amount_plot_df["approved_amount_million_cny"],
    color=amount_colors,
    edgecolor=TOKENS["surface"],
    linewidth=0.8,
    height=0.64,
    zorder=3,
)
ax_amount.set_yticks(y_pos)
ax_amount.set_yticklabels(amount_plot_df["funding_type"].astype(str), fontsize=TICK_LABEL_SIZE, color=BLACK)
ax_amount.set_xlabel("Approved amount (million CNY)", fontsize=AXIS_LABEL_SIZE, labelpad=5, color=BLACK)
add_panel_title(ax_amount, "b", "Approved amount by funding type")
ax_amount.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4, min_n_ticks=3))
ax_amount.grid(axis="x", color=REFERENCE_LINE_COLOR, linewidth=0.75)
ax_amount.grid(axis="y", visible=False)
ax_amount.tick_params(axis="x", labelsize=TICK_LABEL_SIZE, colors=BLACK, length=3, width=0.7)
ax_amount.tick_params(axis="y", labelsize=TICK_LABEL_SIZE, colors=BLACK, length=0, pad=2)
for spine in ["left", "bottom"]:
    ax_amount.spines[spine].set_color(BLACK)
    ax_amount.spines[spine].set_linewidth(AXIS_SPINE_LINEWIDTH)
ax_amount.spines["top"].set_visible(False)
ax_amount.spines["right"].set_visible(False)
amount_x_max = float(amount_plot_df["approved_amount_million_cny"].max()) * 1.18
ax_amount.set_xlim(0, amount_x_max)
ax_amount.spines["left"].set_position(("data", 0))
ax_amount.spines["left"].set_zorder(6)
for bar, value in zip(amount_bars, amount_plot_df["approved_amount_million_cny"]):
    ax_amount.text(
        value + amount_x_max * 0.018,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}",
        ha="left",
        va="center",
        fontsize=VALUE_LABEL_SIZE,
        color=BLACK,
        fontfamily="Times New Roman",
    )
# Running this cell exports Fig. 1 in four formats; later workers decide whether to overwrite output files.
figure_base = FIGURE_DIR / "Fig1_annual_growth"
export_paths = {
    "svg": figure_base.with_suffix(".svg"),
    "pdf": figure_base.with_suffix(".pdf"),
    "png": figure_base.with_suffix(".png"),
    "tiff": figure_base.with_suffix(".tiff"),
}

fig.savefig(export_paths["svg"], facecolor=TOKENS["surface"])
fig.savefig(export_paths["pdf"], facecolor=TOKENS["surface"])
fig.savefig(export_paths["png"], dpi=600, facecolor=TOKENS["surface"])
fig.savefig(export_paths["tiff"], dpi=600, facecolor=TOKENS["surface"], pil_kwargs={"compression": "tiff_lzw"})
plt.show()

export_paths


In [ ]:
# Generate the draft Results paragraph and validation notes.
# The manuscript draft is in English for direct use in the Results section; validation notes retain clear numeric evidence.
peak_row = year_distribution.loc[year_distribution["n"].idxmax()]
peak_year = int(peak_row["award_year"])
peak_n = int(peak_row["n"])
peak_share = float(peak_row["share"] * 100)
peak_period_n = int(year_distribution.loc[year_distribution["award_year"].between(2018, 2020), "n"].sum())
peak_period_share = peak_period_n / total_n * 100
first_year = YEAR_AXIS[0]
last_year = YEAR_AXIS[-1]
first_year_n = int(year_distribution.loc[year_distribution["award_year"].eq(first_year), "n"].iloc[0])
last_year_n = int(year_distribution.loc[year_distribution["award_year"].eq(last_year), "n"].iloc[0])
count_2022 = int(year_distribution.loc[year_distribution["award_year"].eq(2022), "n"].iloc[0])

top_funding_rows = funding_distribution.sort_values("n", ascending=False).head(2).copy()
top_funding_n = int(top_funding_rows["n"].sum())
top_funding_share = top_funding_n / total_n * 100
top_funding_text = " and ".join(
    f"{row.funding_type} (n={int(row.n)}, {float(row.share) * 100:.1f}%)"
    for row in top_funding_rows.itertuples(index=False)
)

top_amount_row = amount_distribution.loc[amount_distribution["approved_amount_million_cny"].idxmax()]
top_amount_type = str(top_amount_row["funding_type"])
top_amount_value = float(top_amount_row["approved_amount_million_cny"])
top_amount_share = float(top_amount_row["amount_share"] * 100)

# Assemble Markdown tables for quick manual checks of annual n, shares, funding-type composition, and amount structure.
summary_for_md = year_distribution[["award_year", "n", "share_label", "raw_year_present", "note"]].copy()
summary_for_md.columns = ["Award year", "n", "Share", "Raw year present", "Note"]
funding_for_md = funding_distribution[["funding_type", "n", "share_label"]].copy()
funding_for_md.columns = ["Funding type", "n", "Share"]
amount_for_md = amount_distribution[["funding_type", "n", "approved_amount_million_cny", "amount_share_label", "median_amount_10k_cny"]].copy()
amount_for_md["approved_amount_million_cny"] = amount_for_md["approved_amount_million_cny"].round(2)
amount_for_md.columns = ["Funding type", "n", "Approved amount (million CNY)", "Amount share", "Median amount (10k CNY)"]

results_paragraph = (
    f"The new master dataset contained {total_n} NSFC awards with valid award-year information across {first_year}-{last_year}. "
    f"Annual records increased from {first_year_n} awards in {first_year} and peaked at {peak_n} awards in {peak_year} "
    f"({peak_share:.2f}% of the analytical sample). "
    f"The 2018-2020 window accounted for {peak_period_n} awards ({peak_period_share:.2f}%). "
    f"The full Fig. 1c x-axis retains every award year through {last_year}, where the source table contains {last_year_n} awards; 2022 contains {count_2022} awards. "
    f"Funding-type composition was dominated by {top_funding_text}, which together represented "
    f"{top_funding_n} awards ({top_funding_share:.1f}%). "
    f"Approved funding totaled {total_approved_amount_million_cny:.2f} million CNY, with {top_amount_type} "
    f"accounting for the largest total ({top_amount_value:.2f} million CNY, {top_amount_share:.1f}%)."
)

log_text = f"""# 03 Year Trend Results Draft

## Validation

- Source file: `data/{DATA_FILENAME}`
- Total deduplicated records: {total_n}
- `award_year` missing values: 0
- Award-year axis: {first_year}-{last_year}
- Raw award years present: {', '.join(map(str, raw_years))}
- 2022 records in source `award_year`: {count_2022}
- Funding-type rule: parsed from `agency_program` before the semicolon and mapped to English NSFC program labels.
- Approved amount rule: `amount_original` is interpreted as `approved_amount_10k_cny`; plotted values are converted to million CNY.
- Fig. 1 design: panel a funding-type donut and panel b approved-amount bars use all {total_n} records; panel c annual bars use the full {first_year}-{last_year} `award_year` axis (n={fig1c_n}); cumulative line removed; no global top title and no bottom footnote.

## Year Distribution

{summary_for_md.to_markdown(index=False)}

## Funding Type Distribution

{funding_for_md.to_markdown(index=False)}

## Approved Amount Distribution

{amount_for_md.to_markdown(index=False)}

## Results paragraph draft

{results_paragraph}

## Figure Caption Draft

Fig. 1. Funding-type composition, approved funding scale and approval-year distribution of the {total_n} included NSFC-funded projects. a, Funding type parsed from the `agency_program` project type before the semicolon. b, Total approved amount by funding type, converted from `amount_original` in 10,000 CNY to million CNY. c, Annual awards across the full {first_year}-{last_year} `award_year` axis.
"""

log_path = LOG_DIR / "03_year_trend_text.md"
log_path.write_text(log_text, encoding="utf-8")

print(results_paragraph)
print(f"Peak year: {peak_year} ({peak_n} awards, {peak_share:.2f}%)")
print(f"Top funding types: {top_funding_text}; combined {top_funding_n} awards ({top_funding_share:.1f}%)")
print(f"Top approved amount type: {top_amount_type} ({top_amount_value:.2f} million CNY, {top_amount_share:.1f}%)")
print(f"Saved table: {table_path}")
print(f"Saved log: {log_path}")


In [ ]:
# Final output validation.
# Check each required file for this task to ensure it exists and is non-empty after a full notebook run.
expected_files = [
    TABLE_DIR / "03_year_distribution.csv",
    LOG_DIR / "03_year_trend_text.md",
    FIGURE_DIR / "Fig1_annual_growth.svg",
    FIGURE_DIR / "Fig1_annual_growth.pdf",
    FIGURE_DIR / "Fig1_annual_growth.tiff",
    FIGURE_DIR / "Fig1_annual_growth.png",
]

file_audit = []
for path in expected_files:
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    file_audit.append({"file": str(path.relative_to(PROJECT_ROOT)), "exists": exists, "bytes": size})
    assert exists and size > 0, f"Expected output missing or empty: {path}"

pd.DataFrame(file_audit)